# Inspect Current ND Experiment — live probe notebook

Interactive probe of the **currently configured ND experiment definition** via raw
NIS macro calls — one small macro per question, each an independent `nis_ar -mw`
run, so a failure in one query doesn't affect the others (a crashing getter
can't take down the rest).

**Open question:** the manual documents the time-phase getters
(`ND_GetTimeLapsePhaseCount`, `ND_GetTimePhaseSchedule`) for the "current
timelapse experiment table", but there is no stimulation-specific phase getter.
Hunch (untested): the sequential stimulation experiment reuses the same
time-phase table — `ND_StimulationAppendPhase` is documented as "appends a time
phase to the current sequential stimulation experiment" — so the time-phase
getters may expose its phases. If confirmed live, this feeds the plan to pick
the `run_stimulation_experiment` timeout from the acquisition settings instead
of the fixed 300 s.

Run **at the microscope workstation** with NIS-Elements open. The read-only
sections come first; the last section is **destructive** (it resets the
configured stimulation phases) — see its warning.

Recorded results belong in `docs/SESSION_HISTORY.md`; confirmed behavior in
`docs/NIS_REFERENCE.md`.


In [ ]:
from autofrap.microscope import nis as nis_util

NIS_EXE = r'C:\Program Files\NIS-Elements\nis_ar.exe'
IN = nis_util.INI_PLACEHOLDER


def ask(label, body, keys, section='q'):
    """Run one small macro; return dict of the requested ini keys ({} on failure)."""
    try:
        cfg = nis_util._run_macro(NIS_EXE, body, ini=True)
        return {k: cfg[section][k] for k in keys}
    except Exception as e:
        print('  %s: FAILED (%s: %s)' % (label, type(e).__name__, e))
        return {}


## Read-only: ND Acquisition tab states

`ND_IsAcqTabChecked("<tab>")` — verified tab names (case sensitive; the
multichannel tab displays as λ): Time, XY, Z, Lambda, Large Image.


In [ ]:
TABS = ['Time', 'XY', 'Z', 'Lambda', 'Large Image']

for tab in TABS:
    res = ask('tab %r' % tab,
              'Int_SetKeyValue("%s","q","on",ND_IsAcqTabChecked("%s"));' % (IN, tab),
              ['on'])
    if res:
        print('  %-12s active=%s' % (tab, res['on']))


## Read-only: time phases — the core question

`ND_GetTimeLapsePhaseCount()` — number of time phases in the current ND
experiment. `ND_GetTimePhaseSchedule(i, &interval, &duration, &loopcnt)` —
per-phase interval [ms], duration [ms], image count. Note: the getter does
**not** return the phase type (acquisition / stimulation / bleaching /
waiting), only timing + loop count.

Not usable here: `ND_GetExperimentLoopSize` queries the *open document* only
(-9 with no document open).


In [ ]:
res = ask('phase count',
          'Int_SetKeyValue("%s","q","phases",ND_GetTimeLapsePhaseCount());' % IN,
          ['phases'])
N_PHASES = int(res['phases']) if res else 0
print('time phases in current ND experiment:', N_PHASES)


In [ ]:
for i in range(N_PHASES):
    body = '''
double interval, duration;
int loopcnt;
ND_GetTimePhaseSchedule(%d, &interval, &duration, &loopcnt);
Int_SetKeyValue("%s","q","interval",interval);
Int_SetKeyValue("%s","q","duration",duration);
Int_SetKeyValue("%s","q","loopcnt",loopcnt);
''' % (i, IN, IN, IN)
    res = ask('phase %d schedule' % i, body, ['interval', 'duration', 'loopcnt'])
    if res:
        print('  phase %d: interval=%s ms  duration=%s ms  loops=%s'
              % (i, res['interval'], res['duration'], res['loopcnt']))
if not N_PHASES:
    print('  (no phases)')


## Read-only: multipoint + Z series

`ND_MP_GetCount()` — multipoint position count. `ND_GetZSeriesExp(...)` —
Z-series settings of the current ND Acquisition definition. Loop params
persist even when the tab is unticked — tab state and params are separate
questions.


In [ ]:
res = ask('position count',
          'Int_SetKeyValue("%s","q","count",ND_MP_GetCount());' % IN, ['count'])
if res:
    print('multipoint positions:', res['count'])

body = '''
int ztype, zcount, zhome_def, zclose;
double ztop, zhome, zbottom, zstep;
char zdevice[256];
char before[256];
char after[256];
ND_GetZSeriesExp(&ztype, &ztop, &zhome, &zbottom, &zstep, &zcount, &zhome_def, &zclose, &zdevice, &before, &after);
Int_SetKeyValue("%s","q","ztype",ztype);
Int_SetKeyValue("%s","q","ztop",ztop);
Int_SetKeyValue("%s","q","zbottom",zbottom);
Int_SetKeyValue("%s","q","zstep",zstep);
Int_SetKeyValue("%s","q","zcount",zcount);
Int_SetKeyString("%s","q","zdevice",zdevice);
''' % (IN, IN, IN, IN, IN, IN)
res = ask('z series', body,
          ['ztype', 'ztop', 'zbottom', 'zstep', 'zcount', 'zdevice'])
if res:
    print('  type=%s top=%s bottom=%s step=%s count=%s device=%r'
          % (res['ztype'], res['ztop'], res['zbottom'], res['zstep'],
             res['zcount'], res['zdevice']))


## Optional: channels — CAUTION

`ND_GetLambdaChannel(i, ...)` **crashed NIS Elements twice** on 20260904 (cause
unidentified; a `done` marker written after the call survived once, the channel
name buffer never came back). Left here commented out — uncomment only if you
accept the risk of a NIS crash.


In [ ]:
# CAUTION: crashed NIS twice on 20260904 — see the note above.
# for i in range(8):
#     body = '''
# char name[256];
# char oc[256];
# char before[256];
# char after[256];
# int color, aftype, afarg1, afarg2;
# strcpy(&name, "SENTINEL");
# ND_GetLambdaChannel(%d, &name, &oc, &color, &before, &after, &aftype, &afarg1, &afarg2);
# Int_SetKeyString("%s","q","name",name);
# Int_SetKeyString("%s","q","oc",oc);
# ''' % (i, IN, IN)
#     res = ask('channel %d' % i, body, ['name', 'oc'])
#     if not res or res['name'] in ('SENTINEL', ''):
#         break
#     print('  channel %d: name=%r oc=%r' % (i, res['name'], res['oc']))


## Stimulation ground truth — DESTRUCTIVE

The cells below call `ND_StimulationResetPhases()`, which **removes all phases
of the current sequential stimulation experiment** — i.e. the GUI template the
pipeline runs (`run_stimulation_experiment` runs "the currently configured"
experiment). Afterwards the template is gone and must be reconfigured in the
GUI. (Per user: fine for one-off exploration.)

**Variant A (read-only, do this first):** with your sequential stimulation
experiment configured in the NIS GUI, re-run the *Read-only: time phases*
cells above and note what the getters report. If they show your stimulation
phases, the hunch is confirmed without any state change.

**Variant B (ground truth, below):** reset the stimulation phases and append
three known phases via `ND_StimulationAppendPhase(type, interval_ms,
duration_ms)` — types: -1 waiting, 0 acquisition, 1 stimulation, 2 bleaching —
then re-query and compare against the known values.


In [ ]:
# Known ground truth appended to the sequential stimulation experiment:
PHASES = [
    (0, 1000.0, 5000.0),   # 0 acquisition, interval 1000 ms, duration 5000 ms
    (1,    0.0,  200.0),   # 1 stimulation, no delay,       duration  200 ms
    (2,    0.0,  500.0),   # 2 bleaching,   no delay,       duration  500 ms
]

body = 'ND_StimulationResetPhases();\n'
for ptype, interval, duration in PHASES:
    body += 'ND_StimulationAppendPhase(%d, %s, %s);\n' % (ptype, interval, duration)
body += 'Int_SetKeyValue("%s","q","done",1);' % IN
res = ask('reset + append known phases', body, ['done'])
print('phases appended:', res)


In [ ]:
res = ask('phase count (after append)',
          'Int_SetKeyValue("%s","q","phases",ND_GetTimeLapsePhaseCount());' % IN,
          ['phases'])
n = int(res['phases']) if res else 0
print('getters report %d time phases (expected %d)' % (n, len(PHASES)))

for i in range(n):
    body = '''
double interval, duration;
int loopcnt;
ND_GetTimePhaseSchedule(%d, &interval, &duration, &loopcnt);
Int_SetKeyValue("%s","q","interval",interval);
Int_SetKeyValue("%s","q","duration",duration);
Int_SetKeyValue("%s","q","loopcnt",loopcnt);
''' % (i, IN, IN, IN)
    r = ask('phase %d schedule' % i, body, ['interval', 'duration', 'loopcnt'])
    print('phase %d:' % i)
    if i < len(PHASES):
        ptype, interval, duration = PHASES[i]
        print('  expected : type=%d interval=%s ms duration=%s ms'
              % (ptype, interval, duration))
    else:
        print('  expected : (out of range)')
    if r:
        print('  getter   : interval=%s ms duration=%s ms loops=%s'
              % (r['interval'], r['duration'], r['loopcnt']))
    else:
        print('  getter   : FAILED')
